# 🧠 Tổng Hợp Kiến Thức CNN (Convolutional Neural Network)

---

## 1. Convolution là gì?

Hãy tưởng tượng bạn có một **tờ giấy kẻ ô** (ảnh) và một **con dấu nhỏ 3×3** (filter/kernel).  
Bạn trượt con dấu qua từng vị trí trên tờ giấy → tại mỗi vị trí **nhân các số tương ứng rồi cộng lại** → ra 1 số duy nhất → tạo thành **feature map**.

```
Ảnh gốc 5×5        Filter 3×3           Feature Map
┌───────────┐      ┌─────────┐
│ 1  2  3.. │  ×   │ 1  0 -1 │   →   Phát hiện đặc trưng
│ 6  7  8.. │      │ 1  0 -1 │
│11 12 13.. │      │ 1  0 -1 │
└───────────┘      └─────────┘
```

---

## 2. Padding

### Không có padding (`VALID`)
- Filter trượt tự do → **output nhỏ hơn input**
- Công thức: `Output = Input - Filter + 1`

```
Input 32×32, Filter 3×3 → Output 30×30  ❌ Mất thông tin viền
```

### Có padding (`SAME`)
- TensorFlow tự thêm **viền 0** xung quanh ảnh
- Output **bằng đúng kích thước input**
- Padding mỗi bên = `(Filter - 1) / 2 = (3-1)/2 = 1 pixel`

```
Input 32×32, Filter 3×3 → Output 32×32  ✅ Giữ nguyên kích thước
```

| | `VALID` | `SAME` |
|---|---|---|
| Thêm pixel 0 | Không | Có |
| Output size | Nhỏ hơn input | Bằng input |
| Công thức | Input - Filter + 1 | Bằng Input |
| Dùng khi nào | Muốn giảm size | Muốn giữ nguyên size |

---

## 3. Strides `[1, 1, 1, 1]`

Bước nhảy của filter khi trượt qua ảnh — **4 chiều vì TensorFlow xử lý batch**:

```
[batch_stride, height_stride, width_stride, channel_stride]
      1,              1,            1,              1
```

| Vị trí | Giá trị | Ý nghĩa |
|---|---|---|
| `[0]` batch | Luôn = 1 | Không bỏ qua ảnh nào trong batch |
| `[1]` height | Tùy chỉnh | Trượt N pixel theo chiều dọc |
| `[2]` width | Tùy chỉnh | Trượt N pixel theo chiều ngang |
| `[3]` channel | Luôn = 1 | Không bỏ qua kênh nào |

```
Stride = 1 → [1,1,1,1]: Chi tiết, output lớn, chậm hơn
Stride = 2 → [1,2,2,1]: Thưa hơn, output nhỏ 2 lần, nhanh hơn
```

---

## 4. Kernel shape `[3, 3, 1, 1]`

```
[filter_height, filter_width, in_channels, out_channels]
      3,              3,           1,            1
```

| Chiều | Ý nghĩa |
|---|---|
| `height, width` | Kích thước filter (3×3, 5×5...) |
| `in_channels` | Phải khớp với số kênh ảnh đầu vào (1=grayscale, 3=RGB) |
| `out_channels` | Số feature map tạo ra (= số filters) |

```python
# Ảnh grayscale + 1 filter  → [3, 3, 1, 1]
# Ảnh RGB     + 32 filters  → [3, 3, 3, 32]
```

> **Lưu ý:** Keras tự động xử lý 4D, chỉ cần khai báo đơn giản:
> ```python
> Conv2D(filters=32, kernel_size=(3,3), strides=(1,1), padding='same')
> ```

---

## 5. Giá trị pixel âm sau Convolution

Pixel gốc **không bao giờ âm** (0–255). Giá trị âm **chỉ xuất hiện sau convolution**.

| Giá trị | Ý nghĩa |
|---|---|
| **Dương lớn** | Đặc trưng khớp với kernel — có cạnh/góc tại đây |
| **Gần 0** | Không có đặc trưng tại vị trí đó |
| **Âm lớn** | Đặc trưng ngược chiều với kernel |

### ReLU loại bỏ giá trị âm
```
f(x) = max(0, x)

-30 → 0    ← Bị loại bỏ
  0 → 0
 15 → 15   ← Giữ lại
 60 → 60   ← Giữ lại
```

**Tại sao loại bỏ?**
- Ảnh không có pixel âm → giá trị âm không có ý nghĩa vật lý
- Giữ tính **phi tuyến** để học pattern phức tạp
- Giảm nhiễu — "không khớp" thì bỏ đi

---

## 6. Output Shape qua từng layer

> Ảnh đầu vào CIFAR-10: **32×32×3** (RGB)

| Layer | Tham số | Output Shape | Ý nghĩa |
|---|---|---|---|
| **Conv2D** | 32 filters, 3×3, relu, same | `(32, 32, 32)` | Phát hiện đặc trưng cơ bản: cạnh, góc, màu |
| **MaxPooling2D** | 2×2 | `(16, 16, 32)` | Giảm kích thước ÷ 2, giữ thông tin quan trọng |
| **Conv2D** | 64 filters, 3×3, relu, same | `(16, 16, 64)` | Học đặc trưng phức tạp hơn |
| **MaxPooling2D** | 2×2 | `(8, 8, 64)` | Giảm kích thước ÷ 2 lần nữa |
| **Conv2D** | 64 filters, 3×3, relu, same | `(8, 8, 64)` | Học đặc trưng sâu hơn |
| **Flatten** | — | `(4096,)` | 8×8×64=4096, chuyển 3D → vector 1D |
| **Dense** | 64 neurons, relu | `(64,)` | Fully connected, học cách phân loại |
| **Dense** | 10 neurons, softmax | `(10,)` | 10 xác suất tương ứng 10 class |

### Quy luật thay đổi shape

| Thao tác | Ảnh hưởng |
|---|---|
| `Conv2D` + `padding='same'` | Giữ nguyên H, W — chỉ thay đổi số kênh = số filters |
| `MaxPooling2D(2×2)` | H ÷ 2, W ÷ 2 — số kênh không đổi |
| `Flatten` | H × W × C → 1 chiều duy nhất |
| `Dense(n)` | Output luôn là `(n,)` bất kể input |

---

## 7. Xử lý ảnh đầu vào

```python
image = tf.io.decode_jpeg(image, channels=1)         # Đọc JPEG → tensor, grayscale
image = tf.image.resize(image, [300, 300])            # Resize về 300×300
image = tf.image.convert_image_dtype(image, tf.float32)  # 0-255 → 0.0-1.0
```

| Bước | Input | Output | Mục đích |
|---|---|---|---|
| `decode_jpeg` | Bytes thô | `(H, W, 1)` | Chuyển file → tensor số |
| `resize` | `(H, W, 1)` | `(300, 300, 1)` | Chuẩn hóa kích thước |
| `convert_dtype` | `0–255` int | `0.0–1.0` float32 | Neural network học tốt hơn với giá trị nhỏ |

---

## 8. Nhận biết Overfitting

| Dấu hiệu | Ý nghĩa |
|---|---|
| Train Accuracy >> Validation Accuracy | Model thuộc lòng data, không generalize được |
| Validation Loss tăng khi Train Loss giảm | Model "học vẹt" |

### Cách khắc phục

```python
# 1. Dropout
model.add(Dropout(0.5))

# 2. Early Stopping
EarlyStopping(monitor='val_loss', patience=3)

# 3. Data Augmentation
ImageDataGenerator(rotation_range=15, horizontal_flip=True)

# 4. L2 Regularization
Dense(64, kernel_regularizer=l2(0.001))
```

---

## 9. Luồng xử lý tổng quát

```
Ảnh đầu vào (32, 32, 3)
        ↓
   Conv2D + ReLU          → Phát hiện đặc trưng cơ bản
        ↓
   MaxPooling              → Giảm kích thước, giữ thông tin
        ↓
   Conv2D + ReLU          → Học đặc trưng phức tạp hơn
        ↓
   MaxPooling              → Giảm kích thước tiếp
        ↓
   Conv2D + ReLU          → Học đặc trưng sâu nhất
        ↓
   Flatten                 → Chuyển 3D → 1D vector
        ↓
   Dense + ReLU           → Học cách phân loại
        ↓
   Dense + Softmax         → 10 xác suất → chọn class cao nhất
```

---

*Tài liệu tổng hợp từ buổi học CNN với CIFAR-10*